In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from PIL import Image
import re
import string
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
import os
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModel
from tqdm import tqdm
from collections import Counter
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import pickle

In [18]:
print("Loading Flickr8k dataset...")
dataset = load_dataset('jxie/flickr8k')
print(f"Dataset structure: {dataset}")
print(f"Dataset keys: {dataset.keys()}")

Loading Flickr8k dataset...
Dataset structure: DatasetDict({
    train: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 6000
    })
    validation: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4'],
        num_rows: 1000
    })
})
Dataset keys: dict_keys(['train', 'validation', 'test'])


In [19]:
print(f"Train set columns: {dataset['train'].column_names}")
print(f"Validation set columns: {dataset['validation'].column_names}")
print(f"Test set columns: {dataset['test'].column_names}")
print(f"\nTrain set size: {len(dataset['train'])}")
print(f"Validation set size: {len(dataset['validation'])}")
print(f"Test set size: {len(dataset['test'])}")
print(f"\nFirst training sample:")
first_sample = dataset['train'][0]
for key in first_sample.keys():
    print(f"{key}: ", end="")
    value = first_sample[key]
    if isinstance(value, Image.Image):
        print(f"Image of size {value.size}")
    else:
        print(f"{value}")

Train set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']
Validation set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']
Test set columns: ['image', 'caption_0', 'caption_1', 'caption_2', 'caption_3', 'caption_4']

Train set size: 6000
Validation set size: 1000
Test set size: 1000

First training sample:
image: Image of size (500, 399)
caption_0: A black dog is running after a white dog in the snow .
caption_1: Black dog chasing brown dog through snow
caption_2: Two dogs chase each other across the snowy ground .
caption_3: Two dogs play together in the snow .
caption_4: Two dogs running through a low lying body of water .


In [21]:
def preprocess_caption(caption):
    caption = caption.lower()
    caption = re.sub(f'[{re.escape(string.punctuation)}]', '', caption)
    caption = ' '.join(caption.split())
    return caption

# Test the preprocessing function on a sample caption
test_caption = "A dog is    running in the park!"
# Call preprocessing function on test caption
cleaned = preprocess_caption(test_caption)
# Print the original caption
print(f"Original: {test_caption}")
# Print the cleaned caption
print(f"Cleaned: {cleaned}")

Original: A dog is    running in the park!
Cleaned: a dog is running in the park


In [22]:
# Define special tokens for the vocabulary
SOS_TOKEN = '<SOS>'  # SOS token marks the start of a sequence
EOS_TOKEN = '<EOS>'  # EOS token marks the end of a sequence
PAD_TOKEN = '<PAD>'  # PAD token is used to pad sequences to the same length
UNK_TOKEN = '<UNK>'  # UNK token represents unknown words not in vocabulary
MAX_VOCAB_SIZE = 10000  # Set maximum vocabulary size to limit memory usage

# Define function to build vocabulary from captions
def build_vocabulary(dataset, max_size=10000):
    word_freq = Counter()  # Initialize counter to count word frequencies
    for split in ['train', 'validation', 'test']:  # Iterate through all captions in train, validation, and test sets
        for sample in tqdm(dataset[split], desc=f"Building vocab from {split}"):
            for i in range(5):   # Iterate through all caption fields (caption_0 to caption_4)
                caption_key = f'caption_{i}'
                caption = sample[caption_key]
                caption = preprocess_caption(caption)
                words = caption.split()
                word_freq.update(words)
    most_common_words = word_freq.most_common(max_size) # Get the most common words up to max_size
    most_common_words = [word for word, freq in most_common_words] # Extract just the words (ignore frequencies)


    word2idx = {  # Create word to index mapping starting from special tokens
        PAD_TOKEN: 0,  # Index 0 is reserved for PAD token
        UNK_TOKEN: 1,  # Index 1 is reserved for UNK token
        SOS_TOKEN: 2,  # Index 2 is reserved for SOS token
        EOS_TOKEN: 3,} # Index 3 is reserved for EOS token
    # Add common words to vocabulary starting from index 4
    for idx, word in enumerate(most_common_words, start=4):
        # Map each word to a unique index
        word2idx[word] = idx
    # Create reverse mapping from index to word
    idx2word = {idx: word for word, idx in word2idx.items()}
    return word2idx, idx2word # Return both mappings

In [23]:
print("Building vocabulary...")
word2idx, idx2word = build_vocabulary(dataset, max_size=MAX_VOCAB_SIZE)
print(f"Vocabulary size: {len(word2idx)}")
print(f"First 20 words: {list(word2idx.items())[:20]}")

Building vocabulary...


Building vocab from test: 100%|██████████| 1000/1000 [00:02<00:00, 374.51it/s]

Vocabulary size: 8724
First 20 words: [('<PAD>', 0), ('<UNK>', 1), ('<SOS>', 2), ('<EOS>', 3), ('a', 4), ('in', 5), ('the', 6), ('on', 7), ('is', 8), ('and', 9), ('dog', 10), ('with', 11), ('man', 12), ('of', 13), ('two', 14), ('white', 15), ('black', 16), ('boy', 17), ('are', 18), ('woman', 19)]


In [24]:
def encode_caption(caption, word2idx, max_length=50):
    tokens = [word2idx[SOS_TOKEN]]  # Start with SOS token index
    caption = preprocess_caption(caption) # Preprocess the caption
    words = caption.split()
    for word in words:
        if word in word2idx: # If word is in vocabulary, add its index
            tokens.append(word2idx[word]) # Add word index to tokens
        else:
            tokens.append(word2idx[UNK_TOKEN])  # If word is not in vocabulary, add UNK token index
    tokens.append(word2idx[EOS_TOKEN])   # Add EOS token index at the end
    if len(tokens) > max_length:  # Truncate or pad to max_length
        tokens = tokens[:max_length]   # Truncate or pad to max_length
    else:
        tokens = tokens + [word2idx[PAD_TOKEN]] * (max_length - len(tokens))
    return np.array(tokens, dtype=np.int64)   # Convert to numpy array and return

In [25]:
# Define custom Dataset class for loading images and captions
class Flickr8kDataset(Dataset):

    def __init__(self, raw_dataset, word2idx, split='train', max_caption_length=50):
        self.dataset = raw_dataset[split] # Store the raw dataset
        self.word2idx = word2idx   # Store word to index mapping
        self.split = split        # Store split name (train/validation/test)
        self.max_caption_length = max_caption_length  # Store maximum caption length for encoding

    # Return the number of samples in the dataset
    def __len__(self):
        return len(self.dataset)

    # Get a single sample by index
    def __getitem__(self, idx):
        sample = self.dataset[idx]
        image = sample['image']
        if image.mode != 'RGB':
            image = image.convert('RGB')
        caption = sample['caption_0']  # Store maximum caption length for encoding
        caption_encoded = encode_caption(caption, self.word2idx, self.max_caption_length) # Encode caption as indices
        return {'image': image, 'caption': caption_encoded, 'caption_text': caption}       # Return image and encoded caption

In [26]:
# Create dataset instances for each split
print("Creating dataset instances...")
train_dataset = Flickr8kDataset(dataset, word2idx, split='train')
val_dataset = Flickr8kDataset(dataset, word2idx, split='validation')
test_dataset = Flickr8kDataset(dataset, word2idx, split='test')
print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

# Get a sample to verify everything works
print("\nSample from training dataset:")
sample = train_dataset[0]
print(f"Image size: {sample['image'].size}")
print(f"Caption shape: {sample['caption'].shape}")
print(f"Caption text: {sample['caption_text']}")
print(f"First 10 caption tokens: {sample['caption'][:10]}")

Creating dataset instances...
Train dataset size: 6000
Validation dataset size: 1000
Test dataset size: 1000

Sample from training dataset:
Image size: (500, 399)
Caption shape: (50,)
Caption text: A black dog is running after a white dog in the snow .
First 10 caption tokens: [  2   4  16  10   8  33 269   4  15  10]


In [27]:
# Set device to GPU if available, otherwise CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
# Load Vision Transformer image processor for preprocessing
print("Loading Vision Transformer...")
image_processor = AutoImageProcessor.from_pretrained('google/vit-base-patch16-224')
vit_model = AutoModel.from_pretrained('google/vit-base-patch16-224', output_hidden_states=True)
vit_model = vit_model.to(device)
vit_model.eval()
print(f"ViT Model: {vit_model.config.model_type}")
print(f"Hidden size (embedding dimension): {vit_model.config.hidden_size}")
for param in vit_model.parameters(): # Disable gradient computation for all parameters
    param.requires_grad = False

Using device: cuda
Loading Vision Transformer...


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ViT Model: vit
Hidden size (embedding dimension): 768


In [28]:
def extract_patch_tokens(images, vit_model, image_processor, device):
    """
    Extract patch token embeddings from Vision Transformer.
    Returns: Tensor of shape [batch_size, num_patches, hidden_size]
    For ViT-base-patch16-224: num_patches = 196, hidden_size = 768
    """
    with torch.no_grad():     # Disable gradient computation for efficiency
        inputs = image_processor(images, return_tensors='pt') # Preprocess images using the ViT image processor
        inputs = {k: v.to(device) for k, v in inputs.items()}  # Move preprocessed inputs to device
        outputs = vit_model(**inputs, output_hidden_states=True) # Pass images through ViT model to get encoder outputs
        # Extract last hidden state (shape: [batch_size, seq_length, hidden_size])
        # seq_length includes CLS token + 196 patch tokens = 197
        last_hidden_state = outputs.last_hidden_state
        # Extract patch tokens only (exclude CLS token at index 0)
        # Shape: [batch_size, 196, 768]
        patch_tokens = last_hidden_state[:, 1:, :]
        return patch_tokens

In [29]:
print("Testing patch token extraction...")
# Get first 2 samples from training dataset for testing
sample_images = [train_dataset[0]['image'], train_dataset[1]['image']]
# Extract patch tokens from sample images
patch_tokens = extract_patch_tokens(sample_images, vit_model, image_processor, device)
print(f"Patch tokens shape: {patch_tokens.shape}")
print(f"Expected shape: [2, 196, 768]")

Testing patch token extraction...
Patch tokens shape: torch.Size([2, 196, 768])
Expected shape: [2, 196, 768]


In [30]:
# Define function to extract CLS token from ViT
def extract_cls_token(images, vit_model, image_processor, device):
    """
    Extract CLS token embedding from Vision Transformer.
    Returns: Tensor of shape [batch_size, 1, hidden_size]
    For ViT-base-patch16-224: hidden_size = 768
    """

    with torch.no_grad():   # Disable gradient computation for efficiency
        inputs = image_processor(images, return_tensors='pt')  # Preprocess images using the ViT image processor
        inputs = {k: v.to(device) for k, v in inputs.items()}   # Move preprocessed inputs to device
        outputs = vit_model(**inputs, output_hidden_states=True) # Pass images through ViT model to get encoder outputs
        # Extract last hidden state (shape: [batch_size, seq_length, hidden_size])
        last_hidden_state = outputs.last_hidden_state
        # Extract CLS token only (index 0)
        # Shape: [batch_size, 1, 768]
        cls_token = last_hidden_state[:, 0:1, :]
        return cls_token

In [31]:
print("Testing CLS token extraction...")
cls_tokens = extract_cls_token(sample_images, vit_model, image_processor, device)
print(f"CLS tokens shape: {cls_tokens.shape}")
print(f"Expected shape: [2, 1, 768]")

Testing CLS token extraction...
CLS tokens shape: torch.Size([2, 1, 768])
Expected shape: [2, 1, 768]


In [32]:
# Define projection layer to map ViT embeddings to decoder dimension
class ViTProjection(nn.Module):
    def __init__(self, input_dim=768, output_dim=512, strategy='patch'):
        super(ViTProjection, self).__init__()  # Call parent class constructor
        self.strategy = strategy  # Store strategy type (patch or cls)
        self.projection = nn.Linear(input_dim, output_dim) # Create linear layer to project from input_dim to output_dim

    # Forward pass through projection layer
    def forward(self, embeddings):
        """
        Project embeddings to decoder dimension.
        Input shape: [batch_size, seq_length, 768]
        Output shape: [batch_size, seq_length, output_dim]
        """
        # Apply linear projection to embeddings
        projected = self.projection(embeddings)
        # Return projected embeddings
        return projected

In [34]:
# Create projection layers for both strategies
print("Creating projection layers...")
DECODER_DIM = 512  # Set decoder dimension (target embedding size)
projection_patch = ViTProjection(input_dim=768, output_dim=DECODER_DIM, strategy='patch') # Create projection layer for patch token strategy
projection_cls = ViTProjection(input_dim=768, output_dim=DECODER_DIM, strategy='cls') # Create projection layer for CLS token strategy
# Move projection layers to device
projection_patch = projection_patch.to(device)
projection_cls = projection_cls.to(device)
# Print projection layer information
print(f"Projection layers created with input_dim=768, output_dim={DECODER_DIM}")

print("\nTesting projection layers...")
projected_patch = projection_patch(patch_tokens)
print(f"Projected patch tokens shape: {projected_patch.shape}")
projected_cls = projection_cls(cls_tokens)
print(f"Projected CLS tokens shape: {projected_cls.shape}")

Creating projection layers...
Projection layers created with input_dim=768, output_dim=512

Testing projection layers...
Projected patch tokens shape: torch.Size([2, 196, 512])
Projected CLS tokens shape: torch.Size([2, 1, 512])


In [37]:
#summary of both strategies
print("SUMMARY:")
print("Strategy A (Patch Tokens):")
print("  - Uses 196 patch token embeddings (one for each 16x16 patch)")
print("  - Preserves spatial information from the image")
print("  - Enables fine-grained cross-attention between image regions and caption words")
print("  - Allows visualization of attention heatmaps as 14x14 grids")
print("  - Higher memory and compute requirements")
print("\nStrategy B (CLS Token):")
print("  - Uses only the global CLS token embedding")
print("  - Loses spatial information from the image")
print("  - Simple baseline with lower memory requirements")
print("  - Cannot visualize spatial attention patterns")
print("  - Better for resource-constrained environments")

SUMMARY:
Strategy A (Patch Tokens):
  - Uses 196 patch token embeddings (one for each 16x16 patch)
  - Preserves spatial information from the image
  - Enables fine-grained cross-attention between image regions and caption words
  - Allows visualization of attention heatmaps as 14x14 grids
  - Higher memory and compute requirements

Strategy B (CLS Token):
  - Uses only the global CLS token embedding
  - Loses spatial information from the image
  - Simple baseline with lower memory requirements
  - Cannot visualize spatial attention patterns
  - Better for resource-constrained environments
